                # 00 Colab Bootstrap And Git Sync

                Purpose: mount Drive, clone or pull the canonical GitHub repo,
                install dependencies, verify imports, inspect the Phase 1 stage
                registry, create Drive folders, and record bootstrap status.

                Phase 1 only. This notebook does not push to GitHub and does
                not run backtesting, execution, risk, or dashboard stages.
                


## 1. Mount Drive And Define Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Clone Or Pull Latest Code From GitHub


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Load Phase 1 Helpers


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Verify Stage Registry


In [ ]:
                specs = list_stage_specs()
                print("Registered stages:", len(specs))
                for spec in specs:
                    print(f"{spec.name:32s} phase={spec.phase:8s} allowed_before_phase2={spec.allowed_before_phase2_promotion}")
                


## 5. Verify Runtime And GPU Availability


In [ ]:
                import importlib
                import pandas as pd
                import numpy as np

                print("pandas:", pd.__version__)
                print("numpy:", np.__version__)
                print("sklearn available:", importlib.util.find_spec("sklearn") is not None)

                try:
                    subprocess.run(["nvidia-smi"], check=False)
                except FileNotFoundError:
                    print("No GPU command found. CPU mode is acceptable for debug runs.")
                


## 6. Create Project Folders


In [ ]:
run_stage_checked("bootstrap")


## 7. Record Sync Stage Status


In [ ]:
run_stage_checked("sync_repo")


## 8. Bootstrap Summary


In [ ]:
                from pathlib import Path

                expected = [
                    "configs/project.yaml",
                    "configs/data.yaml",
                    "notebooks/05_phase1_performance_package_lab.ipynb",
                    "src/btc_quant/pipeline/stages.py",
                    "SKILL.md",
                ]
                for rel in expected:
                    path = Path(PROJECT_ROOT) / rel
                    print("OK" if path.exists() else "MISSING", rel)
                


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
